# 단조 지체 모델 검증 예시

이 노트북은 합성 관측값으로 모델 검증 방식을 보여줍니다. 수치는 졸업 연구 결과나 실제 모델 성능이 아닙니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from event_traffic.modeling import evaluate_regressors_by_group, fit_isotonic_curve

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data = pd.read_csv(root / 'data' / 'sample' / 'delay_observations.csv')
data.head()

## 날짜 전체를 제외해 평가

초기 노트북은 모델을 학습한 자료에서 다시 평가했습니다. 공개 코드는 한 날짜의 모든 관측을 학습에서 제외한 뒤 해당 날짜를 예측합니다. 음수 이동시간 차이도 삭제하지 않습니다.

In [ ]:
validation, predictions = evaluate_regressors_by_group(
    data,
    x_column='excess_trip_index',
    y_column='delay_min',
    group_column='observation_date',
)
validation

비교 모델은 평균 기준선, 선형 회귀, isotonic regression입니다. 실제 자료에서는 날짜뿐 아니라 행정동도 같은 fold에 묶어 지역 누수를 막아야 합니다.

In [ ]:
model = fit_isotonic_curve(data, x_column='excess_trip_index', y_column='delay_min')
grid = np.linspace(data['excess_trip_index'].min(), data['excess_trip_index'].max(), 200)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(data['excess_trip_index'], data['delay_min'], alpha=0.65, label='Synthetic observations')
ax.plot(grid, model.predict(grid), color='#d97706', linewidth=2.5, label='Isotonic fit')
ax.axhline(0, color='#64748b', linewidth=1)
ax.set(xlabel='Excess-trip index', ylabel='Travel-time difference (min)')
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## 다음 검증

원본 데이터 재분석에서는 동일 OD·수단 단위 이동시간 차이, 날짜·지역 grouped validation, 불확실성 구간을 함께 보고합니다. 현재 합성 샘플 결과는 코드가 의도대로 작동하는지만 확인합니다.